# OpenMontage — Chatterbox voice-clone candidate TEST (9 real CC0 voices)
Clones Chatterbox from each real CC0 reference (OwenTyme/voice-zero) and generates
ONE line from that channel's actual script. Human picks finals. Self-reporting.

In [ ]:
import os
import sys
import time
import json
import torch
from pathlib import Path
TEST_FORCE_T4 = True
GPU_BRANCH='unknown'; GPU_NAME='unknown'
if torch.cuda.is_available():
    cc=torch.cuda.get_device_properties(0).major; GPU_NAME=torch.cuda.get_device_name(0)
    if TEST_FORCE_T4:
        if cc<7: print(f'[FATAL] T4 required, got CC={cc}'); sys.exit(1)
        GPU_BRANCH='t4_or_better'
    elif cc>=7: GPU_BRANCH='t4_or_better'
    elif cc==6: GPU_BRANCH='p100'
    else: GPU_BRANCH='old_gpu'
else:
    print('No GPU'); sys.exit(1)
print(f'GPU: {GPU_NAME} (CC={cc}) branch={GPU_BRANCH}')

In [ ]:
import subprocess, sys
def pip_install(no_deps,*pkgs):
    cmd=[sys.executable,'-m','pip','install','-q']
    if no_deps: cmd.append('--no-deps')
    cmd.extend(pkgs)
    print('  $ pip install '+' '.join(pkgs),flush=True)
    return subprocess.call(cmd)==0
pip_install(True,'chatterbox-tts')
pip_install(True,'resemble-perth>=1.0.0','conformer==0.3.2','spacy-pkuseg','pykakasi==2.3.0','pyloudnorm','omegaconf','s3tokenizer','librosa==0.11.0','gradio==6.8.0')
pip_install(True,'transformers==5.2.0','diffusers==0.29.0')
print('install done')

In [ ]:
import torch, warnings, time, traceback
warnings.filterwarnings('ignore')
device='cuda'
print('Loading Chatterbox (singleton)...',flush=True)
cb=None; cb_sr=24000
try:
    t0=time.time(); from chatterbox.tts import ChatterboxTTS
    cb=ChatterboxTTS.from_pretrained(device=device); cb_sr=int(cb.sr)
    print(f'  loaded in {round(time.time()-t0,2)}s sr={cb_sr}',flush=True)
except Exception as e:
    print('load FAILED',repr(e)); traceback.print_exc(); sys.exit(1)

In [ ]:
BASE=None  # set below
CANDIDATES = [
  {
    "channel": "crime-ledger",
    "name": "david_wales",
    "url": "https://raw.githubusercontent.com/OwenTyme/voice-zero/main/voices/david_wales.flac",
    "source": "LibriVox reader 6454 — Five Tales by John Galsworthy, Track 1",
    "test_line": "February first, 2009. Jeju Island. A childcare teacher boards a taxi at three AM."
  },
  {
    "channel": "crime-ledger",
    "name": "cori_samuel",
    "url": "https://raw.githubusercontent.com/OwenTyme/voice-zero/main/voices/cori_samuel.flac",
    "source": "LibriVox reader 92 — Black Beauty (version 2), Track 1",
    "test_line": "February first, 2009. Jeju Island. A childcare teacher boards a taxi at three AM."
  },
  {
    "channel": "crime-ledger",
    "name": "simon_evers",
    "url": "https://raw.githubusercontent.com/OwenTyme/voice-zero/main/voices/simon_evers.flac",
    "source": "LibriVox reader 1255 — Celebration of Dialects and Accents Vol 2, Track 18 (English RP)",
    "test_line": "February first, 2009. Jeju Island. A childcare teacher boards a taxi at three AM."
  },
  {
    "channel": "mythology-slavic",
    "name": "padraig_o'hiceadha-lyrical",
    "url": "https://raw.githubusercontent.com/OwenTyme/voice-zero/main/voices/padraig_o'hiceadha-lyrical.flac",
    "source": "LibriVox reader 2588 — Celebration of Dialects and Accents Vol 1, Track 2 (Irish, lyrical)",
    "test_line": "Deep in the Russian forest, something watches. Something ancient. Something conscious."
  },
  {
    "channel": "mythology-slavic",
    "name": "caden_vaughn_clegg-gravel",
    "url": "https://raw.githubusercontent.com/OwenTyme/voice-zero/main/voices/caden_vaughn_clegg-gravel.flac",
    "source": "LibriVox reader 6574 — Frankenstein (version 3), Track 13 (gravelly)",
    "test_line": "Deep in the Russian forest, something watches. Something ancient. Something conscious."
  },
  {
    "channel": "mythology-slavic",
    "name": "andy",
    "url": "https://raw.githubusercontent.com/OwenTyme/voice-zero/main/voices/andy.flac",
    "source": "LibriVox reader 2262 — Zadig or the Book of Fate, Track 11 (Scottish)",
    "test_line": "Deep in the Russian forest, something watches. Something ancient. Something conscious."
  },
  {
    "channel": "speculative-biology",
    "name": "greg_golding",
    "url": "https://raw.githubusercontent.com/OwenTyme/voice-zero/main/voices/greg_golding.flac",
    "source": "LibriVox reader 8222 — History of England..., Track 64 (American)",
    "test_line": "In the Australian outback, a beetle defends itself with chemistry. Explosive chemistry. One milligram"
  },
  {
    "channel": "speculative-biology",
    "name": "nicholas_james_bridgewater",
    "url": "https://raw.githubusercontent.com/OwenTyme/voice-zero/main/voices/nicholas_james_bridgewater.flac",
    "source": "LibriVox reader 1618 — Celebration of Dialects Vol 2, Track 13 (English Mid-Atlantic)",
    "test_line": "In the Australian outback, a beetle defends itself with chemistry. Explosive chemistry. One milligram"
  },
  {
    "channel": "speculative-biology",
    "name": "clayton_j_smith",
    "url": "https://raw.githubusercontent.com/OwenTyme/voice-zero/main/voices/clayton_j_smith.flac",
    "source": "LibriVox reader 487 — Paradise Lost, Track 9 (American)",
    "test_line": "In the Australian outback, a beetle defends itself with chemistry. Explosive chemistry. One milligram"
  }
]
LICENSE = "CC0 1.0 Universal (public domain dedication) \u2014 OwenTyme/voice-zero, sourced from LibriVox (public-domain reading)"
TEST_LINES = {
  "crime-ledger": "February first, 2009. Jeju Island. A childcare teacher boards a taxi at three AM.",
  "mythology-slavic": "Deep in the Russian forest, something watches. Something ancient. Something conscious.",
  "speculative-biology": "In the Australian outback, a beetle defends itself with chemistry. Explosive chemistry. One milligram"
}
OUT = Path('/kaggle/working/clone_test'); OUT.mkdir(parents=True, exist_ok=True)
import urllib.request
def download(url, path):
    req=urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
    data=urllib.request.urlopen(req, timeout=60).read()
    open(path,'wb').write(data)
    return len(data)
def to_wav(flac_path, wav_path, sr=24000):
    import torchaudio as ta
    w,sr0=ta.load(flac_path)
    if w.shape[0]>1: w=w.mean(0,keepdim=True)
    if sr0!=sr: w=ta.functional.resample(w,sr0,sr)
    ta.save(str(wav_path), w, sr)
rows=[]
for c in CANDIDATES:
    cdir=OUT/c['channel']; cdir.mkdir(parents=True, exist_ok=True)
    ref_flac=cdir/(c['name']+'.flac'); ref_wav=cdir/(c['name']+'_ref.wav')
    test_wav=cdir/(c['name']+'_test.wav')
    rec={'channel':c['channel'],'name':c['name'],'source':c['source'],'license':LICENSE,'test_line':c['test_line'],'ref_url':c['url']}
    try:
        download(c['url'], ref_flac)
        to_wav(ref_flac, ref_wav)
        t0=time.time(); wav=cb.generate(c['test_line'], audio_prompt_path=str(ref_wav))
        if torch.cuda.is_available(): torch.cuda.synchronize()
        gen=time.time()-t0; dur=int(wav.shape[-1])/cb_sr
        import torchaudio as ta; ta.save(str(test_wav), wav.cpu(), cb_sr)
        rec.update({'ok':True,'dur_s':round(dur,3),'gen_s':round(gen,3),'test_wav':str(test_wav),'ref_wav':str(ref_wav),'ref_bytes':ref_flac.stat().st_size})
        print(f"  [{c['channel']}] {c['name']}: gen={gen:.2f}s dur={dur:.2f}s",flush=True)
    except Exception as e:
        rec.update({'ok':False,'error':repr(e)})
        print(f"  [{c['channel']}] {c['name']} FAIL: {e}",flush=True)
    rows.append(rec)
manifest={'project_id':'voice-clone-candidate-test','gpu_name':GPU_NAME,'gpu_branch':GPU_BRANCH,
          'voice':'cloned-per-candidate','generated_at':time.strftime('%Y-%m-%dT%H:%M:%S'),'candidates':rows}
(OUT/'clone_test_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))